# 04.1 Origin Classification with Transformer Fine-Tuning

In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
from transformers.utils.notebook import NotebookProgressCallback

In [ ]:
RANDOM_STATE = 42
MIN_SAMPLES_PER_CLASS = 100
MODEL_CHECKPOINT = 'roberta-base'
TEXT_COLUMN = 'text_basic_plus_aspects'
MAX_LENGTH = 256

# Training config — the key changes vs. 04.1
NUM_TRAIN_EPOCHS = 12                     # was 4
TRAIN_BATCH_SIZE = 16                     # was 8
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2                      # effective batch = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06                       # shorter warmup for longer runs
EARLY_STOP_PATIENCE = 3
USE_FP16 = True                           # flip to False if loss becomes NaN

# LR sweep
LR_SWEEP = [1e-5, 2e-5, 3e-5, 5e-5]

OUTPUT_DIR_ROOT = 'artifacts/origin_finetuning_roberta_fixed'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

## Load & preprocess

Same filtering rules, same stratified 70/15/15 split, same `text_basic_plus_aspects` column. This keeps the comparison against 04.1 and against the TF-IDF baseline clean.

In [3]:
df = pd.read_csv('Data/final_coffee_reviews.csv')
print('Shape:', df.shape)

def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['text_basic'] = df['Blind Assessment'].fillna('').map(normalize_text)

for col in ['aroma_text', 'flavor_text', 'acidity_text', 'body_text', 'combined_text']:
    if col not in df.columns:
        df[col] = ''
    df[col] = df[col].fillna('').astype(str)

df['aspect_text_combined'] = df['combined_text'].map(normalize_text)
df['text_basic_plus_aspects'] = (
    df['text_basic'].fillna('') + ' ' + df['aspect_text_combined'].fillna('')
).str.replace(r'\s+', ' ', regex=True).str.strip()

df['origin_country'] = df['Country'].astype('string').str.strip()
df['origin_country'] = df['origin_country'].replace('', pd.NA)

work = df[(df['text_raw_minimal'].str.len() >= 30) & (df['origin_country'].notna())].copy()
counts = work['origin_country'].value_counts()
valid_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index
work = work[work['origin_country'].isin(valid_classes)].copy().reset_index(drop=True)

print('Rows after filtering:', len(work))
print('Num classes:', work['origin_country'].nunique())
work['origin_country'].value_counts()

Shape: (7585, 30)
Rows after filtering: 6820
Num classes: 15


origin_country
Ethiopia         1996
Colombia          988
Kenya             699
Guatemala         561
Indonesia         452
Costa Rica        373
Panama            354
United States     302
El Salvador       240
Brazil            200
Rwanda            176
Peru              130
Nicaragua         125
Honduras          123
Mexico            101
Name: count, dtype: Int64

In [4]:
y = work['origin_country']
row_idx = work.index

train_idx, temp_idx, y_train, y_temp = train_test_split(
    row_idx, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', len(train_idx), 'Val:', len(val_idx), 'Test:', len(test_idx))

label_names = sorted(work['origin_country'].unique())
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

train_texts = work.loc[train_idx, TEXT_COLUMN].fillna('').tolist()
val_texts   = work.loc[val_idx, TEXT_COLUMN].fillna('').tolist()
test_texts  = work.loc[test_idx, TEXT_COLUMN].fillna('').tolist()

train_labels = work.loc[train_idx, 'origin_country'].map(label2id).tolist()
val_labels   = work.loc[val_idx,   'origin_country'].map(label2id).tolist()
test_labels  = work.loc[test_idx,  'origin_country'].map(label2id).tolist()

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(label_names)),
    y=np.array(train_labels),
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float)
print('Class weights (min/max):', class_weights_np.min().round(3), class_weights_np.max().round(3))

Train: 4774 Val: 1023 Test: 1023
Class weights (min/max): 0.228 4.483


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class CoffeeOriginDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = CoffeeOriginDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset   = CoffeeOriginDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH)
test_dataset  = CoffeeOriginDataset(test_texts,  test_labels,  tokenizer, MAX_LENGTH)
print('Train/Val/Test:', len(train_dataset), len(val_dataset), len(test_dataset))

Train/Val/Test: 4774 1023 1023


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def build_training_args(output_dir: str, learning_rate: float):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        logging_strategy='epoch',
        disable_tqdm=True,
        report_to='none',
        fp16=USE_FP16 and torch.cuda.is_available(),
        seed=RANDOM_STATE,
    )

def run_one(learning_rate: float, use_class_weights: bool, tag: str):
    set_seed(RANDOM_STATE)
    out_dir = os.path.join(OUTPUT_DIR_ROOT, tag)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
    )
    args = build_training_args(out_dir, learning_rate)
    trainer_cls = WeightedTrainer if use_class_weights else Trainer
    extra = dict(class_weights=class_weights_tensor) if use_class_weights else {}
    trainer = trainer_cls(
        model=model, args=args,
        train_dataset=train_dataset, eval_dataset=val_dataset,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
        **extra,
    )
    try:
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass
    print(f'\n=== Training {tag} | lr={learning_rate} | weighted={use_class_weights} ===')
    train_out = trainer.train()
    val_metrics = trainer.evaluate(eval_dataset=val_dataset)
    test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix='test')
    return trainer, train_out, val_metrics, test_metrics

## Step 1 — LR sweep with plain (unweighted) cross-entropy

The goal here is to first verify the model can actually learn. If any LR drives training loss well below `ln(15) ≈ 2.708` and validation Macro-F1 above the TF-IDF baseline (0.134), then the 04.1 failure was a training-config issue, not a data issue.

In [7]:
sweep_results = []
for lr in LR_SWEEP:
    tag = f'plainCE_lr{lr:.0e}'
    _, train_out, val_m, test_m = run_one(lr, use_class_weights=False, tag=tag)
    sweep_results.append({
        'tag': tag,
        'lr': lr,
        'final_train_loss': train_out.training_loss,
        'val_f1_macro': val_m['eval_f1_macro'],
        'val_balanced_acc': val_m['eval_balanced_accuracy'],
        'val_accuracy': val_m['eval_accuracy'],
        'test_f1_macro': test_m['test_f1_macro'],
        'test_balanced_acc': test_m['test_balanced_accuracy'],
        'test_accuracy': test_m['test_accuracy'],
    })

sweep_df = pd.DataFrame(sweep_results).sort_values('val_f1_macro', ascending=False)
sweep_df

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | Details
--------------------------------+------------+--------
lm_head.layer_norm.bias         | UNEXPECTED |        
lm_head.bias                    | UNEXPECTED |        
lm_head.dense.bias              | UNEXPECTED |        
lm_head.dense.weight            | UNEXPECTED |        
roberta.embeddings.position_ids | UNEXPECTED |        
lm_head.layer_norm.weight       | UNEXPECTED |        
classifier.out_proj.weight      | MISSING    |        
classifier.dense.weight         | MISSING    |        
classifier.out_proj.bias        | MISSING    |        
classifier.dense.bias           | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated a


=== Training plainCE_lr1e-05 | lr=1e-05 | weighted=False ===
{'loss': '4.861', 'grad_norm': '12.66', 'learning_rate': '9.764e-06', 'epoch': '1'}
{'eval_loss': '2.3', 'eval_accuracy': '0.2933', 'eval_balanced_accuracy': '0.06667', 'eval_precision_macro': '0.01955', 'eval_recall_macro': '0.06667', 'eval_f1_macro': '0.03023', 'eval_runtime': '1.406', 'eval_samples_per_second': '727.4', 'eval_steps_per_second': '22.75', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '4.5', 'grad_norm': '9.429', 'learning_rate': '8.877e-06', 'epoch': '2'}
{'eval_loss': '2.21', 'eval_accuracy': '0.3236', 'eval_balanced_accuracy': '0.1049', 'eval_precision_macro': '0.07844', 'eval_recall_macro': '0.1049', 'eval_f1_macro': '0.07401', 'eval_runtime': '1.241', 'eval_samples_per_second': '824.4', 'eval_steps_per_second': '25.79', 'epoch': '2'}


...

There were unexpected keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.beta', 'roberta.embeddings.LayerNorm.gamma', 'roberta.encoder.layer.0.attention.output.LayerNorm.beta', 'roberta.encoder.layer.0.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.0.output.LayerNorm.beta', 'roberta.encoder.layer.0.output.LayerNorm.gamma', 'roberta.encoder.layer.1.attention.output.LayerNorm.beta', 'roberta.encoder.layer.1.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.1.output.LayerNorm.beta', 'roberta.encoder.layer.1.output.LayerNorm.gamma', 'roberta.encoder.layer.2.attention.output.LayerNorm.beta', 'roberta.encoder.layer.2.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.2.output.LayerNorm.beta', 'roberta.encoder.layer.2.output.LayerNorm.gamma', 'roberta.encoder.layer.3.attention.output.LayerNorm.beta', 'roberta.encoder.layer.3.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.3.output.LayerNorm.beta', 'roberta.encoder.layer.3.output.LayerNorm.g

{'eval_loss': '2.331', 'eval_accuracy': '0.2933', 'eval_balanced_accuracy': '0.06667', 'eval_precision_macro': '0.01955', 'eval_recall_macro': '0.06667', 'eval_f1_macro': '0.03023', 'eval_runtime': '1.493', 'eval_samples_per_second': '685.1', 'eval_steps_per_second': '21.43', 'epoch': '4'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '2.333', 'test_accuracy': '0.2923', 'test_balanced_accuracy': '0.06667', 'test_precision_macro': '0.01949', 'test_recall_macro': '0.06667', 'test_f1_macro': '0.03016', 'test_runtime': '1.317', 'test_samples_per_second': '776.4', 'test_steps_per_second': '24.29', 'epoch': '4'}


,tag,lr,final_train_loss,val_f1_macro,val_balanced_acc,val_accuracy,test_f1_macro,test_balanced_acc,test_accuracy
2,plainCE_lr3e-05,0.00003,3.276758,0.181486,0.187649,0.350929,0.164361,0.174847,0.352884
1,plainCE_lr2e-05,0.00002,3.566520,0.164377,0.174513,0.359726,0.158272,0.171164,0.364614
0,plainCE_lr1e-05,0.00001,3.774636,0.144540,0.161936,0.352884,0.151584,0.170846,0.377322
3,plainCE_lr5e-05,0.00005,4.669088,0.030234,0.066667,0.293255,0.030156,0.066667,0.292278


In [8]:
best_row = sweep_df.iloc[0]
BEST_LR = float(best_row['lr'])
print(f'Best LR (by val macro-F1): {BEST_LR}')
print(f'  val macro-F1:  {best_row["val_f1_macro"]:.4f}')
print(f'  test macro-F1: {best_row["test_f1_macro"]:.4f}')
print(f'  test bal-acc:  {best_row["test_balanced_acc"]:.4f}')
print(f'  TF-IDF baseline (from notebook 02): 0.134 macro-F1')

Best LR (by val macro-F1): 3e-05
  val macro-F1:  0.1815
  test macro-F1: 0.1644
  test bal-acc:  0.1748
  TF-IDF baseline (from notebook 02): 0.134 macro-F1


## Step 2 — Rerun best LR with class-weighted CE

Now that we know the model can learn, we re-introduce class weighting to see if it improves macro-F1 on tail classes without collapsing performance on head classes.

In [9]:
weighted_trainer, weighted_train_out, weighted_val_m, weighted_test_m = run_one(
    BEST_LR, use_class_weights=True, tag=f'weightedCE_lr{BEST_LR:.0e}'
)

final_summary = pd.DataFrame([
    {
        'setup': f'plain CE @ lr={BEST_LR}',
        'val_f1_macro': best_row['val_f1_macro'],
        'val_bal_acc': best_row['val_balanced_acc'],
        'test_f1_macro': best_row['test_f1_macro'],
        'test_bal_acc': best_row['test_balanced_acc'],
        'test_accuracy': best_row['test_accuracy'],
    },
    {
        'setup': f'weighted CE @ lr={BEST_LR}',
        'val_f1_macro': weighted_val_m['eval_f1_macro'],
        'val_bal_acc': weighted_val_m['eval_balanced_accuracy'],
        'test_f1_macro': weighted_test_m['test_f1_macro'],
        'test_bal_acc': weighted_test_m['test_balanced_accuracy'],
        'test_accuracy': weighted_test_m['test_accuracy'],
    },
])
final_summary

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | Details
--------------------------------+------------+--------
lm_head.layer_norm.bias         | UNEXPECTED |        
lm_head.bias                    | UNEXPECTED |        
lm_head.dense.bias              | UNEXPECTED |        
lm_head.dense.weight            | UNEXPECTED |        
roberta.embeddings.position_ids | UNEXPECTED |        
lm_head.layer_norm.weight       | UNEXPECTED |        
classifier.out_proj.weight      | MISSING    |        
classifier.dense.weight         | MISSING    |        
classifier.out_proj.bias        | MISSING    |        
classifier.dense.bias           | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated a


=== Training weightedCE_lr3e-05 | lr=3e-05 | weighted=True ===
{'loss': '5.411', 'grad_norm': '5.057', 'learning_rate': '2.927e-05', 'epoch': '1'}
{'eval_loss': '2.709', 'eval_accuracy': '0.06745', 'eval_balanced_accuracy': '0.07804', 'eval_precision_macro': '0.009934', 'eval_recall_macro': '0.07804', 'eval_f1_macro': '0.01708', 'eval_runtime': '1.249', 'eval_samples_per_second': '819', 'eval_steps_per_second': '25.62', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '5.41', 'grad_norm': '3.837', 'learning_rate': '2.661e-05', 'epoch': '2'}
{'eval_loss': '2.709', 'eval_accuracy': '0.1026', 'eval_balanced_accuracy': '0.06667', 'eval_precision_macro': '0.006843', 'eval_recall_macro': '0.06667', 'eval_f1_macro': '0.01241', 'eval_runtime': '1.277', 'eval_samples_per_second': '801.2', 'eval_steps_per_second': '25.06', 'epoch': '2'}


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '2.704', 'eval_accuracy': '0.2502', 'eval_balanced_accuracy': '0.09783', 'eval_precision_macro': '0.03108', 'eval_recall_macro': '0.09783', 'eval_f1_macro': '0.04462', 'eval_runtime': '1.261', 'eval_samples_per_second': '811', 'eval_steps_per_second': '25.37', 'epoch': '7'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '2.699', 'test_accuracy': '0.2307', 'test_balanced_accuracy': '0.08292', 'test_precision_macro': '0.02845', 'test_recall_macro': '0.08292', 'test_f1_macro': '0.04012', 'test_runtime': '1.355', 'test_samples_per_second': '755.3', 'test_steps_per_second': '23.63', 'epoch': '7'}


,setup,val_f1_macro,val_bal_acc,test_f1_macro,test_bal_acc,test_accuracy
0,plain CE @ lr=3e-05,0.181486,0.187649,0.164361,0.174847,0.352884
1,weighted CE @ lr=3e-05,0.044618,0.097830,0.040115,0.082917,0.230694


## Step 3 — Diagnostics on the best model

Pick whichever setup won on val macro-F1 and produce a per-class report plus confusion matrix.

In [10]:
# Choose winner by val macro-F1
if weighted_val_m['eval_f1_macro'] >= best_row['val_f1_macro']:
    winning_trainer = weighted_trainer
    winning_name = f'weighted CE @ lr={BEST_LR}'
else:
    # Re-train best plain-CE to get its trainer object in scope
    winning_trainer, _, _, _ = run_one(BEST_LR, use_class_weights=False, tag=f'plainCE_best_rerun_lr{BEST_LR:.0e}')
    winning_name = f'plain CE @ lr={BEST_LR}'

print(f'Winning setup: {winning_name}')

val_pred_output  = winning_trainer.predict(val_dataset)
test_pred_output = winning_trainer.predict(test_dataset)
val_pred_ids  = np.argmax(val_pred_output.predictions, axis=1)
test_pred_ids = np.argmax(test_pred_output.predictions, axis=1)

val_true_labels  = [id2label[i] for i in val_labels]
test_true_labels = [id2label[i] for i in test_labels]
val_pred_labels  = [id2label[i] for i in val_pred_ids]
test_pred_labels = [id2label[i] for i in test_pred_ids]

print('\n== Test headline ==')
print(f'Macro-F1:          {f1_score(test_true_labels, test_pred_labels, average="macro", zero_division=0):.4f}')
print(f'Balanced Accuracy: {balanced_accuracy_score(test_true_labels, test_pred_labels):.4f}')
print(f'Accuracy:          {accuracy_score(test_true_labels, test_pred_labels):.4f}  (biased by imbalance)')

class_order = work['origin_country'].value_counts().index.tolist()
print('\nTest per-class report (sorted by support):')
print(classification_report(test_true_labels, test_pred_labels, labels=class_order, zero_division=0, digits=3))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | Details
--------------------------------+------------+--------
lm_head.layer_norm.bias         | UNEXPECTED |        
lm_head.bias                    | UNEXPECTED |        
lm_head.dense.bias              | UNEXPECTED |        
lm_head.dense.weight            | UNEXPECTED |        
roberta.embeddings.position_ids | UNEXPECTED |        
lm_head.layer_norm.weight       | UNEXPECTED |        
classifier.out_proj.weight      | MISSING    |        
classifier.dense.weight         | MISSING    |        
classifier.out_proj.bias        | MISSING    |        
classifier.dense.bias           | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated a


=== Training plainCE_best_rerun_lr3e-05 | lr=3e-05 | weighted=False ===
{'loss': '4.745', 'grad_norm': '15.88', 'learning_rate': '2.929e-05', 'epoch': '1'}
{'eval_loss': '2.279', 'eval_accuracy': '0.2717', 'eval_balanced_accuracy': '0.06978', 'eval_precision_macro': '0.02681', 'eval_recall_macro': '0.06978', 'eval_f1_macro': '0.03873', 'eval_runtime': '1.277', 'eval_samples_per_second': '800.8', 'eval_steps_per_second': '25.05', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '4.334', 'grad_norm': '7.633', 'learning_rate': '2.663e-05', 'epoch': '2'}
{'eval_loss': '2.158', 'eval_accuracy': '0.3431', 'eval_balanced_accuracy': '0.1184', 'eval_precision_macro': '0.1392', 'eval_recall_macro': '0.1184', 'eval_f1_macro': '0.09953', 'eval_runtime': '1.243', 'eval_samples_per_second': '823.1', 'eval_steps_per_second': '25.75', 'epoch': '2'}


...

{'train_runtime': '356.5', 'train_samples_per_second': '160.7', 'train_steps_per_second': '5.049', 'train_loss': '3.256', 'epoch': '12'}


There were unexpected keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.beta', 'roberta.embeddings.LayerNorm.gamma', 'roberta.encoder.layer.0.attention.output.LayerNorm.beta', 'roberta.encoder.layer.0.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.0.output.LayerNorm.beta', 'roberta.encoder.layer.0.output.LayerNorm.gamma', 'roberta.encoder.layer.1.attention.output.LayerNorm.beta', 'roberta.encoder.layer.1.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.1.output.LayerNorm.beta', 'roberta.encoder.layer.1.output.LayerNorm.gamma', 'roberta.encoder.layer.2.attention.output.LayerNorm.beta', 'roberta.encoder.layer.2.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.2.output.LayerNorm.beta', 'roberta.encoder.layer.2.output.LayerNorm.gamma', 'roberta.encoder.layer.3.attention.output.LayerNorm.beta', 'roberta.encoder.layer.3.attention.output.LayerNorm.gamma', 'roberta.encoder.layer.3.output.LayerNorm.beta', 'roberta.encoder.layer.3.output.LayerNorm.g

{'eval_loss': '2.408', 'eval_accuracy': '0.3539', 'eval_balanced_accuracy': '0.1795', 'eval_precision_macro': '0.2792', 'eval_recall_macro': '0.1795', 'eval_f1_macro': '0.1771', 'eval_runtime': '1.564', 'eval_samples_per_second': '654.1', 'eval_steps_per_second': '20.46', 'epoch': '12'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '2.35', 'test_accuracy': '0.3675', 'test_balanced_accuracy': '0.1849', 'test_precision_macro': '0.225', 'test_recall_macro': '0.1849', 'test_f1_macro': '0.1783', 'test_runtime': '1.334', 'test_samples_per_second': '766.9', 'test_steps_per_second': '23.99', 'epoch': '12'}
Winning setup: plain CE @ lr=3e-05

== Test headline ==
Macro-F1:          0.1783
Balanced Accuracy: 0.1849
Accuracy:          0.3675  (biased by imbalance)

Test per-class report (sorted by support):
               precision    recall  f1-score   support

     Ethiopia      0.454     0.632     0.529       299
     Colombia      0.254     0.318     0.282       148
        Kenya      0.491     0.533     0.511       105
    Guatemala      0.231     0.405     0.294        84
    Indonesia      0.529     0.529     0.529        68
   Costa Rica      0.088     0.054     0.067        56
       Panama      0.160     0.075     0.103        53
United States      0.167     0.022     0.039        45
  El Salvador    

In [ ]:
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=class_order)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(
    ax=ax, xticks_rotation=90, colorbar=False,
)
plt.title(f'Test Confusion Matrix — {winning_name}')
plt.tight_layout()
plt.show()

In [12]:
results_path = os.path.join(OUTPUT_DIR_ROOT, 'sweep_results.json')
with open(results_path, 'w') as f:
    json.dump({
        'sweep': sweep_results,
        'weighted_best': {
            'lr': BEST_LR,
            'val_f1_macro': weighted_val_m['eval_f1_macro'],
            'test_f1_macro': weighted_test_m['test_f1_macro'],
            'test_balanced_accuracy': weighted_test_m['test_balanced_accuracy'],
        },
        'winner': winning_name,
        'tfidf_baseline_f1_macro': 0.134,
    }, f, indent=2)
print('Saved:', results_path)

Saved: artifacts/origin_finetuning_roberta_fixed\sweep_results.json


## How to interpret the output

- **Training loss should drop clearly below 2.70.** If it stays near 2.70 across all LRs, the problem is not LR — it's the data or the model head.
- **At least one LR should beat the TF-IDF baseline (test Macro-F1 > 0.134).** If not, that is a negative result worth reporting honestly: transformers did not help on this dataset at this scale.
- **Weighted CE usually lifts macro-F1 a little and hurts accuracy.** If weighted CE collapses (macro-F1 ≈ 0.06 again), the class weights are destabilizing training — halve them or switch strategies (e.g., focal loss).

## Step 4 — Weighted CE at lower LRs (fix for collapse)

Weighted CE at lr=5e-5 collapsed (Macro-F1 ≈ 0.012, balanced acc = 1/15 → predicting a single class). High LR × strong class weights destabilized training. Retry at lr=1e-5 and 2e-5.

In [13]:
weighted_retry_results = []
for lr in [1e-5, 2e-5]:
    tag = f'weightedCE_retry_lr{lr:.0e}'
    _, train_out, val_m, test_m = run_one(lr, use_class_weights=True, tag=tag)
    weighted_retry_results.append({
        'tag': tag,
        'lr': lr,
        'final_train_loss': train_out.training_loss,
        'val_f1_macro': val_m['eval_f1_macro'],
        'val_balanced_acc': val_m['eval_balanced_accuracy'],
        'val_accuracy': val_m['eval_accuracy'],
        'test_f1_macro': test_m['test_f1_macro'],
        'test_balanced_acc': test_m['test_balanced_accuracy'],
        'test_accuracy': test_m['test_accuracy'],
    })

weighted_retry_df = pd.DataFrame(weighted_retry_results).sort_values('val_f1_macro', ascending=False)
weighted_retry_df

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | Details
--------------------------------+------------+--------
lm_head.layer_norm.bias         | UNEXPECTED |        
lm_head.bias                    | UNEXPECTED |        
lm_head.dense.bias              | UNEXPECTED |        
lm_head.dense.weight            | UNEXPECTED |        
roberta.embeddings.position_ids | UNEXPECTED |        
lm_head.layer_norm.weight       | UNEXPECTED |        
classifier.out_proj.weight      | MISSING    |        
classifier.dense.weight         | MISSING    |        
classifier.out_proj.bias        | MISSING    |        
classifier.dense.bias           | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated a


=== Training weightedCE_retry_lr1e-05 | lr=1e-05 | weighted=True ===
{'loss': '5.409', 'grad_norm': '5.459', 'learning_rate': '9.758e-06', 'epoch': '1'}
{'eval_loss': '2.708', 'eval_accuracy': '0.08211', 'eval_balanced_accuracy': '0.06667', 'eval_precision_macro': '0.005474', 'eval_recall_macro': '0.06667', 'eval_f1_macro': '0.01012', 'eval_runtime': '1.278', 'eval_samples_per_second': '800.6', 'eval_steps_per_second': '25.04', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '5.378', 'grad_norm': '9.941', 'learning_rate': '8.871e-06', 'epoch': '2'}
{'eval_loss': '2.658', 'eval_accuracy': '0.1486', 'eval_balanced_accuracy': '0.1173', 'eval_precision_macro': '0.1304', 'eval_recall_macro': '0.1173', 'eval_f1_macro': '0.06317', 'eval_runtime': '1.246', 'eval_samples_per_second': '821.1', 'eval_steps_per_second': '25.68', 'epoch': '2'}


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '2.677', 'eval_accuracy': '0.2375', 'eval_balanced_accuracy': '0.214', 'eval_precision_macro': '0.1879', 'eval_recall_macro': '0.214', 'eval_f1_macro': '0.1782', 'eval_runtime': '1.6', 'eval_samples_per_second': '639.3', 'eval_steps_per_second': '20', 'epoch': '12'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '2.731', 'test_accuracy': '0.2268', 'test_balanced_accuracy': '0.1968', 'test_precision_macro': '0.1901', 'test_recall_macro': '0.1968', 'test_f1_macro': '0.1669', 'test_runtime': '1.323', 'test_samples_per_second': '773.2', 'test_steps_per_second': '24.18', 'epoch': '12'}


,tag,lr,final_train_loss,val_f1_macro,val_balanced_acc,val_accuracy,test_f1_macro,test_balanced_acc,test_accuracy
1,weightedCE_retry_lr2e-05,0.00002,4.060733,0.178220,0.214001,0.237537,0.166871,0.196847,0.226784
0,weightedCE_retry_lr1e-05,0.00001,4.877433,0.149701,0.190620,0.212121,0.145798,0.189279,0.221896


## Step 5 — Combined results table and save

In [14]:
combined_rows = []
combined_rows.append({'model': 'TF-IDF (baseline, nb 02)', 'setup': '-', 'val_f1_macro': None, 'test_f1_macro': 0.134, 'test_bal_acc': None, 'test_acc': None})
combined_rows.append({'model': 'RoBERTa (04.1 original)', 'setup': 'weighted CE lr=2e-5 4ep', 'val_f1_macro': 0.0583, 'test_f1_macro': 0.0612, 'test_bal_acc': 0.0973, 'test_acc': 0.2776})

for r in sweep_results:
    combined_rows.append({'model': 'RoBERTa', 'setup': r['tag'], 'val_f1_macro': r['val_f1_macro'], 'test_f1_macro': r['test_f1_macro'], 'test_bal_acc': r['test_balanced_acc'], 'test_acc': r['test_accuracy']})

for r in weighted_retry_results:
    combined_rows.append({'model': 'RoBERTa', 'setup': r['tag'], 'val_f1_macro': r['val_f1_macro'], 'test_f1_macro': r['test_f1_macro'], 'test_bal_acc': r['test_balanced_acc'], 'test_acc': r['test_accuracy']})

combined_df = pd.DataFrame(combined_rows).sort_values('test_f1_macro', ascending=False, na_position='last').reset_index(drop=True)
print(combined_df.to_string())

with open(os.path.join(OUTPUT_DIR_ROOT, 'all_results.json'), 'w') as f:
    json.dump({
        'roberta_sweep': sweep_results,
        'roberta_weighted_retry': weighted_retry_results,
        'tfidf_baseline_f1_macro': 0.134,
    }, f, indent=2)
    
print('Saved all_results.json')

                      model                     setup  val_f1_macro  test_f1_macro  test_bal_acc  test_acc
0                   RoBERTa  weightedCE_retry_lr2e-05      0.178220       0.166871      0.196847  0.226784
1                   RoBERTa           plainCE_lr3e-05      0.181486       0.164361      0.174847  0.352884
2                   RoBERTa           plainCE_lr2e-05      0.164377       0.158272      0.171164  0.364614
3                   RoBERTa           plainCE_lr1e-05      0.144540       0.151584      0.170846  0.377322
4                   RoBERTa  weightedCE_retry_lr1e-05      0.149701       0.145798      0.189279  0.221896
5  TF-IDF (baseline, nb 02)                         -           NaN       0.134000           NaN       NaN
6   RoBERTa (04.1 original)   weighted CE lr=2e-5 4ep      0.058300       0.061200      0.097300  0.277600
7                   RoBERTa           plainCE_lr5e-05      0.030234       0.030156      0.066667  0.292278
Saved all_results.json
